In [0]:
%sql
CREATE OR REPLACE VIEW sts_prd.`01_ipd_thering`.g_bowler_ww_inventory_all_ipd AS
WITH material_set AS (
  SELECT DISTINCT Product, Level1Desc
  FROM hub_prd.g_bpc_finance.v_reltioproduct_hier
  WHERE Level1Desc IN (
    'Alaris System','Hazardous Drug Solutions','IV Solutions',
    'Gravity and Syringe Delivery','IV Access','CME',
    'Alaris Plus','IPD OEM','BD Nexus'
  )
  AND Product IS NOT NULL
),
fy_bounds AS (
  SELECT
    add_months(date_trunc('year', add_months(current_date, -9)),  9) AS fy_start_curr, -- Oct 1 of current FY
    -- add_months(date_trunc('year', add_months(current_date, -9)), -3) AS fy_start_prev, -- Oct 1 of previous FY
    add_months(date_trunc('year', add_months(current_date, -9)), 21) AS fy_start_next  -- Oct 1 of next FY
),
inventory_ipd AS (
  SELECT so.*
  FROM hub_prd.s_isc.ihd_snapshot_daily AS so
  CROSS JOIN fy_bounds b
  WHERE so.SnapshotDate >= b.fy_start_curr  -- include Current FY
    AND so.SnapshotDate <  b.fy_start_next   -- up to (but not including) next FY
)
SELECT
  i.*,
  ms.Level1Desc
FROM inventory_ipd AS i
INNER JOIN material_set AS ms
  ON i.MaterialID = ms.Product;
